In [36]:
# imports and setup

import os
import base64
import sys
from dotenv import load_dotenv
from typing import Any, Dict, List, Optional, Tuple
from PIL import Image   # noqa: F401
from pydantic import BaseModel as PydanticBaseModel
from pydantic import Field
from pydantic_settings import BaseSettings
import oracledb
import io
import cv2
import fitz
import numpy as np
import boto3
from botocore.config import Config
import json
import re
import time
from tqdm import tqdm
import mariadb
import pandas as pd

# Add project root to Python path so we can import from app module
project_root = os.path.dirname(os.getcwd())
if project_root not in sys.path:
    sys.path.append(project_root)
# Now we can import from app (after adding to sys.path)
from app.utils.logger import get_logger
# Load environment variables from the project root directory
env_path = os.path.join(project_root, '.env')
load_dotenv(env_path)

logger = get_logger(name=__name__)

## database connections and data extraction

In [2]:

class AppConstants:
    """Constantes globais para a aplicação."""

    BEDROCK_DEFAULT_MODEL_ID = "anthropic.claude-3-7-sonnet-20240729-v1:0"
    # DEFAULT_PROMPT_EXTRACAO_PATH = "prompt_extracao_laudo.txt"
    # DEFAULT_PROMPT_RESUMO_PATH = "prompt_resumo.txt"
    # S3_BUCKET_NAME = "agente-ai-laudos"
    # S3_RESULTS_PREFIX = "resultados"
    # S3_DEBUG_PREFIX = "debug"
    MAX_RETRIES = 5
    INITIAL_BACKOFF_SECONDS = 2


In [3]:
class Settings(BaseSettings):
    """Carrega e valida as configurações a partir de variáveis de ambiente."""

    ORACLE_USER: str
    ORACLE_PASSWORD: str
    ORACLE_DSN: str
    ORACLE_INSTANT_CLIENT_PATH: Optional[str] = Field(
        None, alias="oracle_instant_client_path"
    )
    AWS_ACCESS_KEY_ID: str
    AWS_SECRET_ACCESS_KEY: str
    AWS_BEDROCK_REGION: str
    BEDROCK_MODEL_ID: str = AppConstants.BEDROCK_DEFAULT_MODEL_ID
    MARIADB_USER: str
    MARIADB_PASSWORD: str
    MARIADB_HOST: str
    MARIADB_PORT: int = 3306
    MARIADB_DATABASE: str
    API_BASE_URL: Optional[str] = Field(None, alias="api_base_url")
    API_USERNAME: Optional[str] = Field(None, alias="username")
    API_PASSWORD: Optional[str] = Field(None, alias="password")

    class Config:
        env_file = ".env"
        env_file_encoding = "utf-8"


In [4]:
def criar_boto3_client(
    service_name: str, settings: Settings, config: Optional[Config] = None
) -> boto3.client:
    try:
        logger.info(
            f"Criando cliente {service_name.upper()} para a região: {settings.AWS_BEDROCK_REGION}..."
        )
        client = boto3.client(
            service_name,
            region_name=settings.AWS_BEDROCK_REGION,
            aws_access_key_id=settings.AWS_ACCESS_KEY_ID,
            aws_secret_access_key=settings.AWS_SECRET_ACCESS_KEY,
            config=config,
        )
        logger.info(f"Cliente {service_name.upper()} criado com sucesso.")
        return client
    except Exception as e:
        logger.critical(f"Não foi possível criar o cliente {service_name.upper()}: {e}")
        raise


In [5]:
def load_query_from_file(file_path: str) -> str:
    """
    Load SQL query from a text file
    
    Args:
        file_path: Path to the query file
    
    Returns:
        Query string
    """
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            content = file.read()
            
        # Extract just the query part (remove variable assignment)
        if '"""' in content:
            # Find the query between triple quotes
            start = content.find('"""') + 3
            end = content.rfind('"""')
            query = content[start:end].strip()
        else:
            # If no triple quotes, assume entire file is the query
            query = content.strip()
            
        return query
        
    except Exception as e:
        logger.error(f"❌ Error loading query from file: {e}")
        return None

In [6]:
# connection to the database

class OracleService:
    def __init__(self, settings: Settings):
        self.config = {
            "user": settings.ORACLE_USER,
            "password": settings.ORACLE_PASSWORD,
            "dsn": settings.ORACLE_DSN,
        }
        self.connection = None
        if settings.ORACLE_INSTANT_CLIENT_PATH and os.path.isdir(
            settings.ORACLE_INSTANT_CLIENT_PATH
        ):
            logger.info(
                f"Inicializando Oracle Client de: {settings.ORACLE_INSTANT_CLIENT_PATH}"
            )
            oracledb.init_oracle_client(lib_dir=settings.ORACLE_INSTANT_CLIENT_PATH)

    def __enter__(self):
        """Establishes the database connection when entering the 'with' block."""
        try:
            self.connection = oracledb.connect(**self.config)
            logger.info(f"✅ Connection established to {self.config['dsn']}")
            return self
        except Exception as e:
            logger.error(f"❌ Error connecting to Oracle: {e}")
            raise

    def __exit__(self, exc_type, exc_val, exc_tb):
        """Closes the database connection when exiting the 'with' block."""
        if self.connection:
            try:
                self.connection.close()
                logger.info("✅ Oracle connection closed.")
            except Exception as e:
                logger.error(f"❌ Error closing Oracle connection: {e}")
        # If an exception occurred within the 'with' block, it will be re-raised.

    def execute_query(self, query: str, fetch_limit: int = None) -> Optional[List[Dict[str, Any]]]:
        """
        Execute a query using the existing Oracle connection.
        
        Args:
            query: SQL query to execute
            fetch_limit: Maximum number of rows to fetch (None for all)
        
        Returns:
            List of dictionaries containing query results, or None if error
        """
        if not self.connection:
            logger.error("❌ Cannot execute query: Not connected. Use within a 'with' block.")
            return None
            
        try:
            with self.connection.cursor() as cursor:
                logger.info("Executing query...")
                cursor.execute(query)
                
                columns = [desc[0] for desc in cursor.description]
                
                if fetch_limit:
                    rows = cursor.fetchmany(fetch_limit)
                else:
                    rows = cursor.fetchall()
                
                results = []
                for row in rows:
                    row_dict = {}
                    for i, col_val in enumerate(row):
                        col_name = columns[i]
                        if isinstance(col_val, oracledb.LOB):
                            row_dict[col_name] = col_val.read()
                        else:
                            row_dict[col_name] = col_val
                    results.append(row_dict)
                
                logger.info(f"✅ Fetched {len(results)} rows.")
                return results
                    
        except Exception as e:
            logger.error(f"Error executing query: {e}")
            return None

In [8]:
class MariaDBService:
    """Context manager for MariaDB database connections."""
    
    def __init__(self, settings: Settings):
        self.config = {
            "user": settings.MARIADB_USER,
            "password": settings.MARIADB_PASSWORD,
            "host": settings.MARIADB_HOST,
            "port": settings.MARIADB_PORT,
            "database": settings.MARIADB_DATABASE,
        }
        self.connection = None

    def __enter__(self):
        """Establishes the database connection when entering the 'with' block."""
        try:
            self.connection = mariadb.connect(**self.config)
            logger.info(f"✅ MariaDB connection established to {self.config['host']}:{self.config['port']}/{self.config['database']}")
            return self
        except mariadb.Error as e:
            logger.error(f"❌ Error connecting to MariaDB: {e}")
            raise

    def __exit__(self, exc_type, exc_val, exc_tb):
        """Closes the database connection when exiting the 'with' block."""
        if self.connection:
            try:
                self.connection.close()
                logger.info("✅ MariaDB connection closed.")
            except mariadb.Error as e:
                logger.error(f"❌ Error closing MariaDB connection: {e}")
        # If an exception occurred within the 'with' block, it will be re-raised.

    def execute_query(self, query: str, fetch_limit: int = None) -> Optional[List[Dict[str, Any]]]:
        """
        Execute a query using the existing MariaDB connection.
        
        Args:
            query: SQL query to execute
            fetch_limit: Maximum number of rows to fetch (None for all)
        
        Returns:
            List of dictionaries containing query results, or None if error
        """
        if not self.connection:
            logger.error("❌ Cannot execute query: Not connected. Use within a 'with' block.")
            return None
            
        try:
            with self.connection.cursor(dictionary=True) as cursor:
                logger.info("Executing MariaDB query...")
                cursor.execute(query)
                
                if fetch_limit:
                    results = cursor.fetchmany(fetch_limit)
                else:
                    results = cursor.fetchall()
                
                logger.info(f"✅ Fetched {len(results)} rows from MariaDB.")
                return results
                    
        except mariadb.Error as e:
            logger.error(f"Error executing MariaDB query: {e}")
            return None

    

## query execution

In [79]:
# load the numero de carteirinha query file
query_file_path = "/home/joao/projects/company_projects/carteirinha-api/documents/querys/query_numero_carteirinha.txt"
numero_carteirinha_query = load_query_from_file(query_file_path)
if numero_carteirinha_query is None:
    logger.error("❌ Failed to load the numero de carteirinha query.")
else:
    logger.info("✅ Numero de carteirinha query loaded successfully.")
    logger.debug(f"Query content: {numero_carteirinha_query}")

{"timestamp": "2025-08-07T17:35:05", "level": "INFO", "name": "__main__", "message": "✅ Numero de carteirinha query loaded successfully.", "filename": "2317460933.py", "lineno": 7}
{"timestamp": "2025-08-07T17:35:05", "level": "DEBUG", "name": "__main__", "message": "Query content: SELECT\n    ac.cd_aviso_cirurgia,\n    ac.cd_paciente,\n    cart.NR_CARTEIRA,\n    cart.CD_CONVENIO,\n    cart.NM_EMPRESA\nFROM\n    dbamv.aviso_cirurgia ac\nLEFT JOIN\n    dbamv.carteira cart\nON\n    cart.cd_paciente = ac.cd_paciente\nWHERE\n    ac.cd_aviso_cirurgia IN (\n791978, 791981, 791989, 792005, 792017, 792023, 792024, 792034, 792045, 792058, 792061, 792062, 792064, 792077, 792085, 792097, 792099, 792104, 792108, 792109, \n792110, 792112, 792125, 792132, 792140, 792151, 792152, 792154, 792157, 792159, 792167, 792169, 792176, 792183, 792187, 792197, 792199, 792200, 792201, 792207, \n792214, 792215, 792227, 792229, 792236, 792237, 792245, 792257, 792262, 792269 \n)", "filename": "2317460933.py", "lin

In [81]:
# query to get the numero de carteirinha using OracleService context manager

numero_carteirinha = None
try:
    settings = Settings()
    with OracleService(settings) as oracle_service:
        numero_carteirinha = oracle_service.execute_query(numero_carteirinha_query)
    if numero_carteirinha is None:
        logger.error("❌ Failed to execute numero de carteirinha query.")
    else:
        logger.info("✅ Numero de carteirinha query executed successfully.")
        logger.debug(f"Query results: {numero_carteirinha}")

    # get the results into a nice pandas DataFrame
    if numero_carteirinha:
        df_numero_carteirinha = pd.DataFrame(numero_carteirinha)
        logger.info("✅ Numero de carteirinha DataFrame created successfully.")
        logger.debug(f"DataFrame content:\n{df_numero_carteirinha}")
    else:
        logger.error("❌ No results to create DataFrame from numero de carteirinha query.")

except Exception as e:
    logger.error(f"❌ An error occurred while executing the numero de carteirinha query: {e}")

logger.info(df_numero_carteirinha.info())
df_numero_carteirinha.head(15)  
    


{"timestamp": "2025-08-07T17:37:14", "level": "INFO", "name": "__main__", "message": "✅ Connection established to srvhmddb007-otk6s-scan.sbntdb.vcnprod.oraclevcn.com:1521/PRDREPT_OCI", "filename": "1417701281.py", "lineno": 23}
{"timestamp": "2025-08-07T17:37:14", "level": "INFO", "name": "__main__", "message": "Executing query...", "filename": "1417701281.py", "lineno": 56}
{"timestamp": "2025-08-07T17:37:14", "level": "INFO", "name": "__main__", "message": "✅ Fetched 201 rows.", "filename": "1417701281.py", "lineno": 77}
{"timestamp": "2025-08-07T17:37:14", "level": "INFO", "name": "__main__", "message": "✅ Oracle connection closed.", "filename": "1417701281.py", "lineno": 34}
{"timestamp": "2025-08-07T17:37:14", "level": "INFO", "name": "__main__", "message": "✅ Numero de carteirinha query executed successfully.", "filename": "3554136488.py", "lineno": 11}
{"timestamp": "2025-08-07T17:37:14", "level": "DEBUG", "name": "__main__", "message": "Query results: [{'CD_AVISO_CIRURGIA': 791

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 201 entries, 0 to 200
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CD_AVISO_CIRURGIA  201 non-null    int64  
 1   CD_PACIENTE        201 non-null    int64  
 2   NR_CARTEIRA        198 non-null    object 
 3   CD_CONVENIO        198 non-null    float64
 4   NM_EMPRESA         5 non-null      object 
dtypes: float64(1), int64(2), object(2)
memory usage: 8.0+ KB


,CD_AVISO_CIRURGIA,CD_PACIENTE,NR_CARTEIRA,CD_CONVENIO,NM_EMPRESA
0,791978,282123,11049687,302.0,None
1,791978,282123,12066256,302.0,None
2,791978,282123,12066256,302.0,None
3,791981,1568370,88888018713860012,110.0,None
4,791981,1568370,88888018713860012,110.0,None
5,791981,1568370,88888018713860012,110.0,None
6,791981,1568370,079871184,10.0,None
7,791981,1568370,88888018713860012,110.0,None
8,791989,622610,010018574011,46.0,None
9,791989,622610,00010001006877012,46.0,None


In [42]:
# Load the carteirinha query from file
query_file_path = "/home/joao/projects/company_projects/carteirinha-api/documents/querys/query_carteirinha.txt"
convenio_carteirinha_query = load_query_from_file(query_file_path)

if convenio_carteirinha_query:
    logger.info("✅ Carteirinha query loaded successfully from file!")
else:
    logger.error("❌ Failed to load query from file")

{"timestamp": "2025-08-07T16:23:56", "level": "INFO", "name": "__main__", "message": "✅ Carteirinha query loaded successfully from file!", "filename": "2412977378.py", "lineno": 6}


In [58]:
#  Carteirinha Query using the OracleService context manager

logger.info("🚀 Testing carteirinha query with context manager...")
logger.info("=" * 50)

results = None
try:
    settings = Settings()
    
    # Use the service as a context manager
    with OracleService(settings) as oracle_service:
        if convenio_carteirinha_query:
            results = oracle_service.execute_query(convenio_carteirinha_query, fetch_limit=999)

            if results:
                logger.info(f"\n✅ Query executed successfully! Found {len(results)} sample records")        
                logger.info(f"\n📊 To get all records, run the query without fetch_limit.")
                
            else:
                logger.error("❌ No results returned or query failed inside 'with' block")
        else:
            logger.error("❌ Query not loaded - cannot execute")

except Exception as e:
    logger.error(f"❌ An error occurred outside the 'with' block: {e}")

logger.info("=" * 50)

# If results were fetched, convert to DataFrame
if results:
    df_carteirinha = pd.DataFrame(results)
    logger.info("✅ Carteirinha DataFrame created successfully.")
    logger.debug(f"DataFrame content:\n{df_carteirinha}")
else:
    logger.error("❌ No results to create DataFrame from carteirinha query.")
# Display the DataFrame info and head
logger.info(df_carteirinha.info())
df_carteirinha.head()

{"timestamp": "2025-08-07T16:36:09", "level": "INFO", "name": "__main__", "message": "🚀 Testing carteirinha query with context manager...", "filename": "776376075.py", "lineno": 3}
{"timestamp": "2025-08-07T16:36:09", "level": "INFO", "name": "__main__", "message": "==================================================", "filename": "776376075.py", "lineno": 4}
{"timestamp": "2025-08-07T16:36:10", "level": "INFO", "name": "__main__", "message": "✅ Connection established to srvhmddb007-otk6s-scan.sbntdb.vcnprod.oraclevcn.com:1521/PRDREPT_OCI", "filename": "1417701281.py", "lineno": 23}
{"timestamp": "2025-08-07T16:36:10", "level": "INFO", "name": "__main__", "message": "Executing query...", "filename": "1417701281.py", "lineno": 56}
{"timestamp": "2025-08-07T16:36:41", "level": "INFO", "name": "__main__", "message": "✅ Fetched 30 rows.", "filename": "1417701281.py", "lineno": 77}
{"timestamp": "2025-08-07T16:36:41", "level": "INFO", "name": "__main__", "message": "\n✅ Query executed succes

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 9 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   CD_AVISO_CIRURGIA             30 non-null     int64         
 1   CD_DOCUMENTO_ANEXO_CIRURGICO  30 non-null     int64         
 2   CD_GUIA                       30 non-null     int64         
 3   TP_GUIA                       30 non-null     object        
 4   TP_SITUACAO                   30 non-null     object        
 5   LO_DOCUMENTO_ANEXO_CIRURGICO  30 non-null     object        
 6   DS_EXTENSAO                   30 non-null     object        
 7   DT_ANEXO                      30 non-null     datetime64[ns]
 8   DS_DOCUMENTO_ANEXO            30 non-null     object        
dtypes: datetime64[ns](1), int64(3), object(5)
memory usage: 2.2+ KB


,CD_AVISO_CIRURGIA,CD_DOCUMENTO_ANEXO_CIRURGICO,CD_GUIA,TP_GUIA,TP_SITUACAO,LO_DOCUMENTO_ANEXO_CIRURGICO,DS_EXTENSAO,DT_ANEXO,DS_DOCUMENTO_ANEXO
0,848690,2809795,19259447,I,S,b'%PDF-1.7\n%\x81\x81\x81\x81\n\n5 0 obj\n<<\n...,.pdf,2025-06-13 14:20:00,Carteira do convênio
1,860303,2882344,19567915,I,N,b'%PDF-1.7\n%\x81\x81\x81\x81\n\n4 0 obj\n<<\n...,.pdf,2025-07-17 12:24:46,Carteira do convênio
2,860406,2883132,19570514,I,A,b'%PDF-1.7\n%\x81\x81\x81\x81\n\n4 0 obj\n<<\n...,.pdf,2025-07-17 15:57:24,Carteira do convênio
3,859362,2875665,19544665,I,A,b'%PDF-1.7\n%\x81\x81\x81\x81\n\n4 0 obj\n<<\n...,.pdf,2025-07-15 11:17:50,Carteira do convênio
4,859372,2875710,19544738,I,S,b'%PDF-1.7\n%\x81\x81\x81\x81\n\n4 0 obj\n<<\n...,.pdf,2025-07-15 11:21:55,Carteira do convênio


In [74]:
# aviso cirurgia query file
aviso_cirurgia_query_file = "/home/joao/projects/company_projects/carteirinha-api/documents/querys/query_aviso_cirurgia.txt"
if os.path.exists(aviso_cirurgia_query_file):
    aviso_cirurgia_query = load_query_from_file(aviso_cirurgia_query_file)
    if aviso_cirurgia_query:
        logger.info("✅ Aviso cirurgia query loaded successfully from file!")
    else:
        logger.error("❌ Failed to load aviso cirurgia query from file")
else:
    logger.error(f"❌ Aviso cirurgia query file not found: {aviso_cirurgia_query_file}")

{"timestamp": "2025-08-07T17:28:47", "level": "INFO", "name": "__main__", "message": "✅ Aviso cirurgia query loaded successfully from file!", "filename": "3187017494.py", "lineno": 6}


In [75]:
# avisos query using maria db context manager
logger.info("🚀 Testing avisos query with MariaDB context manager...")
logger.info("=" * 50)
try:
    settings = Settings()
    
    # Use the service as a context manager
    with MariaDBService(settings) as maria_service:
        maria_results = maria_service.execute_query(aviso_cirurgia_query, fetch_limit=50)
        if maria_results:
            logger.info(f"\n✅ Avisos query executed successfully! Found {len(maria_results)} sample records")
            logger.info("\n📋 Sample Results:")
            logger.info("-" * 50)
            
            for i, record in enumerate(maria_results, 1):
                logger.info(f"\nRecord {i}:")
                for key, value in record.items():
                    logger.info(f"  {key}: {value}")
                    
            logger.info(f"\n📊 To get all records, run the query without fetch_limit.")
            
        else:
            logger.error("❌ No results returned or query failed inside 'with' block")
except Exception as e:
    logger.error(f"❌ An error occurred while executing avisos query: {e}")

# If results were fetched, convert to DataFrame
if maria_results:
    df_avisos_maria = pd.DataFrame(maria_results)
    # rename the surgical_order_id column to match the Oracle column
    df_avisos_maria.rename(columns={"surgical_order_id": "CD_AVISO_CIRURGIA"}, inplace=True)
    logger.info("✅ Avisos DataFrame created successfully.")
    logger.debug(f"DataFrame content:\n{df_avisos_maria}")
else:
    logger.error("❌ No results to create DataFrame from avisos query.")
# Display the DataFrame info and head
logger.info(df_avisos_maria.info())
df_avisos_maria.head()

{"timestamp": "2025-08-07T17:28:46", "level": "INFO", "name": "__main__", "message": "🚀 Testing avisos query with MariaDB context manager...", "filename": "1175966137.py", "lineno": 2}
{"timestamp": "2025-08-07T17:28:46", "level": "INFO", "name": "__main__", "message": "==================================================", "filename": "1175966137.py", "lineno": 3}
{"timestamp": "2025-08-07T17:28:47", "level": "INFO", "name": "__main__", "message": "✅ MariaDB connection established to srvawsdb002.cow7tj30bxpl.us-east-1.rds.amazonaws.com:3306/", "filename": "2645936225.py", "lineno": 18}
{"timestamp": "2025-08-07T17:28:47", "level": "INFO", "name": "__main__", "message": "Executing MariaDB query...", "filename": "2645936225.py", "lineno": 51}
{"timestamp": "2025-08-07T17:29:02", "level": "INFO", "name": "__main__", "message": "✅ Fetched 50 rows from MariaDB.", "filename": "2645936225.py", "lineno": 59}
{"timestamp": "2025-08-07T17:29:02", "level": "INFO", "name": "__main__", "message": "\

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 22 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   CD_AVISO_CIRURGIA      50 non-null     int64         
 1   hospital_id            50 non-null     int64         
 2   hospitalization_type   50 non-null     object        
 3   patient_hospitalized   50 non-null     object        
 4   friendly_name          50 non-null     object        
 5   filter_name_hospital   50 non-null     object        
 6   doctor_name            50 non-null     object        
 7   specialty              50 non-null     object        
 8   patient_name           50 non-null     object        
 9   with_opme              50 non-null     object        
 10  opme                   29 non-null     object        
 11  procedimento_teste     50 non-null     object        
 12  created_at             50 non-null     datetime64[ns]
 13  expecte

,CD_AVISO_CIRURGIA,hospital_id,hospitalization_type,patient_hospitalized,friendly_name,filter_name_hospital,doctor_name,specialty,patient_name,with_opme,...,created_at,expected_date,status,provider,dt_status,dt_status_format,mes_ano_criacao,health_insurance_code,health_insurance_name,cluster_hub
0,791978,1,elective,Não,Sto Agostinho,Sto Agostinho,THAIS SYRIO AMARAL,GINECOLOGIA E OBSTETRICIA,ANA CAROLINA BELEM RIOS,Com OPME,...,2025-01-01 15:13:37,2025-01-02 06:00:00,Realizada,None,2025-03-20 18:14:41,2025-03-20,2025-01-01,302,COPASS,CLUSTER RMBH/SALVADOR
1,791981,1,urgency,Não,Sto Agostinho,Sto Agostinho,ANA MARCIA DE MIRANDA COTA,GINECOLOGIA E OBSTETRICIA,CLEYTIANE SOARES BARBOSA,Sem OPME,...,2025-01-01 15:29:59,2025-01-01 06:00:00,Realizada,None,2025-01-02 09:46:54,2025-01-02,2025-01-01,110,SUL AMERICA,CLUSTER RMBH/SALVADOR
2,791989,1,urgency,Não,Sto Agostinho,Sto Agostinho,GERALDO XAVIER DE FARIA JUNIOR,UROLOGIA,CRISTINA LABOISSIERE MARTINS,Com OPME,...,2025-01-01 16:30:15,2025-01-01 06:00:00,Realizada,None,2025-01-02 09:59:02,2025-01-02,2025-01-01,46,CEMIG SAUDE,CLUSTER RMBH/SALVADOR
3,792005,1,urgency,Sim,Sto Agostinho,Sto Agostinho,JULIA BIANCHI BRITO,CIRURGIA PEDIATRICA,RN DE MICHELE NICACIO DE JESUS FERREIRA,Sem OPME,...,2025-01-01 18:52:24,2025-01-03 03:00:00,Cancelada,None,2025-01-24 19:06:32,2025-01-24,2025-01-01,387,AMIL S450,CLUSTER RMBH/SALVADOR
4,792017,1,urgency,Não,Sto Agostinho,Sto Agostinho,LEONARDO SOARES LOPES,CIRURGIA GERAL,TATIANE SANTOS DANTAS DE SOUZA,Com OPME,...,2025-01-01 22:16:27,2025-01-02 06:00:00,Cancelada,None,2025-01-06 15:20:55,2025-01-06,2025-01-01,10,AMIL,CLUSTER RMBH/SALVADOR


In [78]:
# list of CD_AVISO_CIRURGIA from the avisos querY mariadb
if maria_results:
    cd_aviso_cirurgia_list = df_avisos_maria["CD_AVISO_CIRURGIA"].tolist()
    logger.info("✅ List of CD_AVISO_CIRURGIA created successfully.")
    logger.debug(f"List content: {cd_aviso_cirurgia_list}")
else:
    logger.error("❌ No results to create list from avisos query.")


output_file_path = "/home/joao/projects/company_projects/carteirinha-api/documents/querys/cd_aviso_cirurgia_list.txt"
try:
    with open(output_file_path, 'w', encoding='utf-8') as file:
        for i, item in enumerate(cd_aviso_cirurgia_list):
            file.write(f"{item}, ")
            if (i + 1) % 20 == 0:  # New line after every 20 items
                file.write("\n")
    logger.info(f"✅ List of CD_AVISO_CIRURGIA saved to {output_file_path}")
except Exception as e:
    logger.error(f"❌ Failed to save list to file: {e}")

{"timestamp": "2025-08-07T17:33:39", "level": "INFO", "name": "__main__", "message": "✅ List of CD_AVISO_CIRURGIA created successfully.", "filename": "3474776747.py", "lineno": 4}
{"timestamp": "2025-08-07T17:33:39", "level": "DEBUG", "name": "__main__", "message": "List content: [791978, 791981, 791989, 792005, 792017, 792023, 792024, 792034, 792045, 792058, 792061, 792062, 792064, 792077, 792085, 792097, 792099, 792104, 792108, 792109, 792110, 792112, 792125, 792132, 792140, 792151, 792152, 792154, 792157, 792159, 792167, 792169, 792176, 792183, 792187, 792197, 792199, 792200, 792201, 792207, 792214, 792215, 792227, 792229, 792236, 792237, 792245, 792257, 792262, 792269]", "filename": "3474776747.py", "lineno": 5}
{"timestamp": "2025-08-07T17:33:39", "level": "INFO", "name": "__main__", "message": "✅ List of CD_AVISO_CIRURGIA saved to /home/joao/projects/company_projects/carteirinha-api/documents/querys/cd_aviso_cirurgia_list.txt", "filename": "3474776747.py", "lineno": 17}


## we have the 3 tables, lets inner join the dataframes on the CD_AVISO_CIRURGIA for the oracle bases, and that same col is surgical_order_id from maria db

In [69]:
# inner join the dataframes on the CD_AVISO_CIRURGIA for the oracle bases, and that same col is surgical_order_id from maria db
logger.info("🚀 Performing inner join on the dataframes...")
df_merged_oracle = pd.merge(df_carteirinha, df_numero_carteirinha, on="CD_AVISO_CIRURGIA", how="inner")
logger.info("✅ Inner join completed successfully.")
# Display the merged DataFrame info and head
logger.info(df_merged_oracle.info())
# merge the avisos DataFrame with the merged oracle DataFrame
logger.info("🚀 Performing inner join with avisos DataFrame...")
df_merged_final = pd.merge(df_merged_oracle, df_avisos_maria, on="CD_AVISO_CIRURGIA", how="inner")
logger.info("✅ Inner join with avisos DataFrame completed successfully.")
# Display the final merged DataFrame info and head
logger.info(df_merged_final.info())
df_merged_final.head()

{"timestamp": "2025-08-07T17:11:20", "level": "INFO", "name": "__main__", "message": "🚀 Performing inner join on the dataframes...", "filename": "4236645270.py", "lineno": 2}
{"timestamp": "2025-08-07T17:11:20", "level": "INFO", "name": "__main__", "message": "✅ Inner join completed successfully.", "filename": "4236645270.py", "lineno": 4}
{"timestamp": "2025-08-07T17:11:20", "level": "INFO", "name": "__main__", "message": "None", "filename": "4236645270.py", "lineno": 6}
{"timestamp": "2025-08-07T17:11:20", "level": "INFO", "name": "__main__", "message": "🚀 Performing inner join with avisos DataFrame...", "filename": "4236645270.py", "lineno": 8}
{"timestamp": "2025-08-07T17:11:20", "level": "INFO", "name": "__main__", "message": "✅ Inner join with avisos DataFrame completed successfully.", "filename": "4236645270.py", "lineno": 10}
{"timestamp": "2025-08-07T17:11:20", "level": "INFO", "name": "__main__", "message": "None", "filename": "4236645270.py", "lineno": 12}


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 89 entries, 0 to 88
Data columns (total 13 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   CD_AVISO_CIRURGIA             89 non-null     int64         
 1   CD_DOCUMENTO_ANEXO_CIRURGICO  89 non-null     int64         
 2   CD_GUIA                       89 non-null     int64         
 3   TP_GUIA                       89 non-null     object        
 4   TP_SITUACAO                   89 non-null     object        
 5   LO_DOCUMENTO_ANEXO_CIRURGICO  89 non-null     object        
 6   DS_EXTENSAO                   89 non-null     object        
 7   DT_ANEXO                      89 non-null     datetime64[ns]
 8   DS_DOCUMENTO_ANEXO            89 non-null     object        
 9   CD_PACIENTE                   89 non-null     int64         
 10  NR_CARTEIRA                   82 non-null     object        
 11  CD_CONVENIO                   82 n

,CD_AVISO_CIRURGIA,CD_DOCUMENTO_ANEXO_CIRURGICO,CD_GUIA,TP_GUIA,TP_SITUACAO,LO_DOCUMENTO_ANEXO_CIRURGICO,DS_EXTENSAO,DT_ANEXO,DS_DOCUMENTO_ANEXO,CD_PACIENTE,...,created_at,expected_date,status,provider,dt_status,dt_status_format,mes_ano_criacao,health_insurance_code,health_insurance_name,cluster_hub
0,848690,2809795,19259447,I,S,b'%PDF-1.7\n%\x81\x81\x81\x81\n\n5 0 obj\n<<\n...,.pdf,2025-06-13 14:20:00,Carteira do convênio,636847,...,2025-06-13 17:19:54,2025-07-20 03:00:00,Revisão,None,2025-07-31 18:18:51,2025-07-31,2025-06-01,20,BRADESCO,CLUSTER RMBH/SALVADOR
1,848690,2809795,19259447,I,S,b'%PDF-1.7\n%\x81\x81\x81\x81\n\n5 0 obj\n<<\n...,.pdf,2025-06-13 14:20:00,Carteira do convênio,636847,...,2025-06-13 17:19:54,2025-07-20 03:00:00,Revisão,None,2025-07-31 18:18:51,2025-07-31,2025-06-01,20,BRADESCO,CLUSTER RMBH/SALVADOR
2,848690,2809795,19259447,I,S,b'%PDF-1.7\n%\x81\x81\x81\x81\n\n5 0 obj\n<<\n...,.pdf,2025-06-13 14:20:00,Carteira do convênio,636847,...,2025-06-13 17:19:54,2025-07-20 03:00:00,Revisão,None,2025-07-31 18:18:51,2025-07-31,2025-06-01,20,BRADESCO,CLUSTER RMBH/SALVADOR
3,848690,2809795,19259447,I,S,b'%PDF-1.7\n%\x81\x81\x81\x81\n\n5 0 obj\n<<\n...,.pdf,2025-06-13 14:20:00,Carteira do convênio,636847,...,2025-06-13 17:19:54,2025-07-20 03:00:00,Revisão,None,2025-07-31 18:18:51,2025-07-31,2025-06-01,20,BRADESCO,CLUSTER RMBH/SALVADOR
4,860303,2882344,19567915,I,N,b'%PDF-1.7\n%\x81\x81\x81\x81\n\n4 0 obj\n<<\n...,.pdf,2025-07-17 12:24:46,Carteira do convênio,2212222,...,2025-07-17 15:24:19,2025-08-15 03:00:00,Negado,SURGICAL-ORDER-FINALIZATION,2025-07-24 16:54:52,2025-07-24,2025-07-01,110,SUL AMERICA,CLUSTER RMBH/SALVADOR


# new task: take all cirurgias 2025 on MARIADB, them take the aviso virurgia codes and filter the others.

## convertions of blobs to optimized images

In [14]:
# image utils
def converter_blob_para_imagens(blob: bytes, extensao: str) -> List[Image.Image]:
    imagens = []
    ext = extensao.lower().strip(".") if extensao else ""
    try:
        if ext == "pdf":
            with fitz.open(stream=blob, filetype="pdf") as pdf_doc:
                logger.info(f"Processando PDF com {len(pdf_doc)} página(s)...")
                for pagina in pdf_doc:
                    pix = pagina.get_pixmap(matrix=fitz.Matrix(3.0, 3.0), alpha=False)
                    imagens.append(Image.open(io.BytesIO(pix.tobytes("png"))))
        elif ext in ["jpg", "jpeg", "png", "bmp"]:
            imagens.append(Image.open(io.BytesIO(blob)))
        else:
            logger.warning(f"Formato de arquivo não suportado: '{ext}'.")
    except Exception as e:
        logger.error(f"Erro ao converter BLOB para imagem (ext: .{ext}): {e}")
    return imagens


def aplicar_clahe(imagem: Image.Image) -> Image.Image:
    try:
        imagem_cv = cv2.cvtColor(np.array(imagem), cv2.COLOR_RGB2BGR)
        imagem_cinza = cv2.cvtColor(imagem_cv, cv2.COLOR_BGR2GRAY)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        return Image.fromarray(clahe.apply(imagem_cinza))
    except Exception:
        return imagem


def imagem_para_base64(imagem: Image.Image) -> str:
    TARGET_BYTES = 3.5 * 1024 * 1024
    if imagem.mode in ("RGBA", "P"):
        imagem = imagem.convert("RGB")
    for quality in range(95, 15, -10):
        buffer = io.BytesIO()
        imagem.save(buffer, format="JPEG", quality=quality)
        if buffer.tell() <= TARGET_BYTES:
            if quality < 95:
                logger.warning(
                    f"Imagem comprimida (qualidade {quality}%) para caber no limite."
                )
            return base64.b64encode(buffer.getvalue()).decode("utf-8")
    raise ValueError(
        f"Não foi possível reduzir a imagem abaixo de {TARGET_BYTES / (1024 * 1024):.1f}MB."
    )


In [15]:
# extract blob data from the results
blobs = []
if results:
    for record in results:
        blob_data = record.get('LO_DOCUMENTO_ANEXO_CIRURGICO')
        if blob_data and isinstance(blob_data, bytes):
            blobs.append(blob_data)
            logger.info(f"✅ Extracted BLOB of size {len(blob_data)} bytes")
        else:
            logger.warning("No BLOB data found in record or data is not bytes")

if blobs:
    logger.info(f"Successfully extracted {len(blobs)} BLOBs into a list.")
else:
    logger.error("Could not extract any BLOBs from the results.")

{"timestamp": "2025-08-07T10:25:13", "level": "INFO", "name": "__main__", "message": "✅ Extracted BLOB of size 506822 bytes", "filename": "395405530.py", "lineno": 8}
{"timestamp": "2025-08-07T10:25:13", "level": "INFO", "name": "__main__", "message": "✅ Extracted BLOB of size 4224600 bytes", "filename": "395405530.py", "lineno": 8}
{"timestamp": "2025-08-07T10:25:13", "level": "INFO", "name": "__main__", "message": "✅ Extracted BLOB of size 261028 bytes", "filename": "395405530.py", "lineno": 8}
{"timestamp": "2025-08-07T10:25:13", "level": "INFO", "name": "__main__", "message": "✅ Extracted BLOB of size 6864055 bytes", "filename": "395405530.py", "lineno": 8}
{"timestamp": "2025-08-07T10:25:13", "level": "INFO", "name": "__main__", "message": "✅ Extracted BLOB of size 216457 bytes", "filename": "395405530.py", "lineno": 8}
{"timestamp": "2025-08-07T10:25:13", "level": "INFO", "name": "__main__", "message": "Successfully extracted 5 BLOBs into a list.", "filename": "395405530.py", "li

In [16]:
# take the list of blobs and convert them to images. them, save them as base64 strings
base64_images = []
if blobs:
    for i, blob in enumerate(blobs):
        logger.info(f"Converting BLOB {i+1}/{len(blobs)} to images...")
        imagens = converter_blob_para_imagens(blob, "pdf")  # Assuming PDF for this example
        # save the images on ../documents/carteirinhas_images/
        if not os.path.exists("../documents/carteirinhas_images/"):
            os.makedirs("../documents/carteirinhas_images/")
        for j, img in enumerate(imagens):
            img_path = f"../documents/carteirinhas_images/blob_{i+1}_image_{j+1}.png"
            img.save(img_path)
            logger.info(f"Saved image {j+1} from BLOB {i+1} to {img_path}")
        logger.info(f"Extracted {len(imagens)} images from BLOB {i+1}.")

        if imagens:
            for j, img in enumerate(imagens):
                logger.info(f"Processing image {j+1} from BLOB {i+1}...")
                img_clahe = aplicar_clahe(img)
                base64_str = imagem_para_base64(img_clahe)
                base64_images.append(base64_str)
                logger.info(f"✅ Converted image {j+1} to base64 string.")
        else:
            logger.warning(f"No images extracted from BLOB {i+1}.")

{"timestamp": "2025-08-07T10:25:13", "level": "INFO", "name": "__main__", "message": "Converting BLOB 1/5 to images...", "filename": "4197694994.py", "lineno": 5}
{"timestamp": "2025-08-07T10:25:13", "level": "INFO", "name": "__main__", "message": "Processando PDF com 1 página(s)...", "filename": "2299205648.py", "lineno": 8}
{"timestamp": "2025-08-07T10:25:12", "level": "INFO", "name": "__main__", "message": "Saved image 1 from BLOB 1 to ../documents/carteirinhas_images/blob_1_image_1.png", "filename": "4197694994.py", "lineno": 13}
{"timestamp": "2025-08-07T10:25:12", "level": "INFO", "name": "__main__", "message": "Extracted 1 images from BLOB 1.", "filename": "4197694994.py", "lineno": 14}
{"timestamp": "2025-08-07T10:25:12", "level": "INFO", "name": "__main__", "message": "Processing image 1 from BLOB 1...", "filename": "4197694994.py", "lineno": 18}
{"timestamp": "2025-08-07T10:25:12", "level": "INFO", "name": "__main__", "message": "✅ Converted image 1 to base64 string.", "filen

In [17]:
base64_images_count = len(base64_images)
if base64_images_count > 0:
    logger.info(f"Successfully converted {base64_images_count} images to base64 strings.")
else:
    logger.warning("No images were converted to base64 strings. Check the BLOB data.")

{"timestamp": "2025-08-07T10:25:16", "level": "INFO", "name": "__main__", "message": "Successfully converted 5 images to base64 strings.", "filename": "1948280603.py", "lineno": 3}


## Creating the antropic modular llm service
### should be used to multiple antropic models if needed

In [18]:
class CarteirinhaExtraida(PydanticBaseModel):
    """Define a estrutura dos dados extraídos da carteirinha."""
    convenio: Optional[str] = Field(None, description="Nome do convênio de saúde.")
    plano: Optional[str] = Field(None, description="Nome do plano de saúde.")
    nome_pessoa: Optional[str] = Field(None, description="Nome completo do titular ou beneficiário.")
    numero_carteirinha: Optional[str] = Field(None, description="O número de identificação da carteirinha.")

In [ ]:
class AntropicLLMService:
    """Encapsula a lógica de chamada ao modelo de linguagem (Bedrock).
    this class is antropic because of the body especification on antropic model version. also could
    be due the invocation method."""

    def __init__(self, bedrock_client: boto3.client, model_id: str, model_version: str, data_model: PydanticBaseModel):
        self.bedrock_client = bedrock_client
        self.model_id = model_id
        self.model_version = model_version
        self.data_model = data_model

    def _extract_json_from_response(self, raw_text: str) -> str:
        """
        Extrai JSON de diferentes formatos de resposta do LLM.
        Tenta múltiplos métodos de extração para maximizar compatibilidade.
        """
        # Method 1: JSON em blocos de código (```json ... ```)
        code_block_match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', raw_text, re.DOTALL)
        if code_block_match:
            logger.info("JSON extraído de bloco de código")
            return code_block_match.group(1).strip()
        
        # Method 2: JSON standalone (sem blocos de código)
        json_match = re.search(r'\{.*\}', raw_text, re.DOTALL)
        if json_match:
            logger.info("JSON extraído diretamente do texto")
            return json_match.group(0).strip()
        
        
        raise ValueError("Nenhum JSON válido encontrado na resposta do LLM")

    def _validate_and_clean_json(self, json_str: str) -> str:
        """
        Valida e limpa a string JSON antes da validação Pydantic.
        """
        try:
            # Tenta fazer parse para verificar se é JSON válido
            parsed = json.loads(json_str)
            # Se chegou até aqui, o JSON é válido
            return json_str
        except json.JSONDecodeError as e:
            logger.warning(f"JSON inválido detectado: {e}")
            # Tenta algumas correções comuns
            cleaned = json_str.strip()
            # Remove possíveis caracteres extras no início/fim
            cleaned = re.sub(r'^[^\{]*', '', cleaned)
            cleaned = re.sub(r'[^\}]*$', '', cleaned)
            try:
                json.loads(cleaned)
                return cleaned
            except json.JSONDecodeError:
                raise ValueError(f"Não foi possível corrigir o JSON: {e}")

    def _invocar_llm(self, corpo_requisicao: Dict, context_log: str = "") -> Dict:
        resultado = {
            "success": False,
            "dados": None,
            "raw_response": "",
            "error": None,
            "input_tokens": 0,
            "output_tokens": 0,
        }
        retries = 0
        while retries < AppConstants.MAX_RETRIES:
            try:
                response = self.bedrock_client.invoke_model(
                    body=json.dumps(corpo_requisicao), modelId=self.model_id
                )
                response_body = json.loads(response.get("body").read())
                usage = response_body.get("usage", {})
                raw_text = response_body.get("content", [{}])[0].get("text", "")
                
                resultado.update({
                    "input_tokens": usage.get("input_tokens", 0),
                    "output_tokens": usage.get("output_tokens", 0),
                    "raw_response": raw_text,
                })
                
                # Extração e validação do JSON
                json_str = self._extract_json_from_response(raw_text)
                cleaned_json = self._validate_and_clean_json(json_str)
                
                resultado["dados"] = cleaned_json
                resultado["success"] = True
                return resultado

            except self.bedrock_client.exceptions.ThrottlingException as e:
                retries += 1
                wait_time = AppConstants.INITIAL_BACKOFF_SECONDS * (2 ** (retries - 1))
                logger.warning(
                    f"LLM Throttling para {context_log}. Tentativa {retries}/{AppConstants.MAX_RETRIES}. "
                    f"Aguardando {wait_time:.2f}s. Erro: {e}"
                )
                time.sleep(wait_time)
            except Exception as e:
                resultado["error"] = str(e)
                logger.error(f"Erro na invocação do LLM para {context_log}: {resultado['error']}")
                return resultado
                
        resultado["error"] = f"Falha no LLM após {AppConstants.MAX_RETRIES} tentativas."
        logger.error(resultado["error"])
        return resultado

    def extrair_dados_de_imagem(self, prompt: str, imagem_base64: str, context_log: str) -> Dict:
        """
        Envia uma imagem e um prompt para o LLM e valida a resposta com o modelo Pydantic.
        """
        corpo = {
            "anthropic_version": self.model_version,
            "max_tokens": 4096,
            "temperature": 0.05,
            "messages": [
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "image",
                            "source": {
                                "type": "base64",
                                "media_type": "image/jpeg",
                                "data": imagem_base64,
                            },
                        },
                        {"type": "text", "text": prompt},
                    ],
                }
            ],
        }
        
        resultado = self._invocar_llm(corpo, context_log)
        
        if resultado["success"]:
            try:
                llm_output_json = resultado["dados"]  # JSON string limpo
                
                # Debug: Log do que estamos tentando validar
                logger.debug(f"Raw LLM response for {context_log}: {resultado['raw_response'][:200]}...")
                logger.debug(f"Extracted JSON for {context_log}: {llm_output_json}")
                
                # Validação Pydantic
                parsed_output = self.data_model.model_validate_json(llm_output_json)
                resultado["dados"] = parsed_output
                
                # Log dos dados extraídos para debug
                logger.info(f"✅ Dados extraídos para {context_log}:")
                logger.info(f"  Convênio: {parsed_output.convenio}")
                logger.info(f"  Plano: {parsed_output.plano}")
                logger.info(f"  Nome: {parsed_output.nome_pessoa}")
                logger.info(f"  Número: {parsed_output.numero_carteirinha}")
                
            except Exception as pydantic_error:
                logger.error(f"❌ Erro de validação Pydantic para {context_log}: {pydantic_error}")
                logger.error(f"JSON problemático: {llm_output_json}")
                logger.error(f"Raw response: {resultado['raw_response']}")
                resultado.update({
                    "success": False,
                    "error": f"Erro de validação Pydantic: {pydantic_error}",
                })
        
        return resultado

## experimenting with one or more models

In [28]:
class ExtractionExperiment:
    """
    Encapsula a lógica para executar um experimento de extração de dados
    usando um modelo e prompt específicos.
    """
    def __init__(self, settings: Settings, model_id: str, model_version: str, prompt: str, prompt_name: str, data_model: PydanticBaseModel):
        self.settings = settings
        self.model_id = model_id
        self.model_version = model_version
        self.prompt = prompt
        self.prompt_name = prompt_name
        self.data_model = data_model
        self.llm_service = self._initialize_llm_service()
        self.results = []

    def _initialize_llm_service(self) -> Optional[AntropicLLMService]:
        """Inicializa o serviço LLM para o modelo especificado."""
        try:
            bedrock_client = criar_boto3_client("bedrock-runtime", self.settings)
            llm_service = AntropicLLMService(bedrock_client=bedrock_client, model_id=self.model_id, model_version=self.model_version, data_model=self.data_model)
            logger.info(f"✅ Serviço LLM inicializado para o modelo: {self.model_id}")
            return llm_service
        except Exception as e:
            logger.critical(f"❌ Falha ao inicializar o serviço LLM para {self.model_id}: {e}")
            return None

    def run(self, images_base64: List[str]):
        """Executa o experimento de extração nas imagens fornecidas."""
        if not self.llm_service:
            logger.error("Não é possível executar o experimento, o serviço LLM não foi inicializado.")
            return

        logger.info(f"🚀 Executando experimento com o modelo '{self.model_id}' e prompt '{self.prompt_name}'...")

        for i, b64_image in tqdm(enumerate(images_base64)):
            context_log = f"imagem_{i+1}"
            logger.info(f"--- Processando {context_log} ---")
            
            extraction_result = self.llm_service.extrair_dados_de_imagem(
                prompt=self.prompt,
                imagem_base64=b64_image,
                context_log=context_log
            )
            
            if extraction_result["success"]:
                dados = extraction_result["dados"]
                logger.info("✅ Extração bem-sucedida!")
                self.results.append(dados.model_dump())
            else:
                logger.error(f"❌ Falha na extração para {context_log}: {extraction_result['error']}")
        
        logger.info(f"✅ Experimento finalizado. {len(self.results)} extrações bem-sucedidas.")
        self.save_results()

    def save_results(self):
        """Salva os resultados da extração em um arquivo JSON com nome dinâmico."""
        if not self.results:
            logger.warning("⚠️ Nenhum resultado para salvar.")
            return

        model_name_safe = self.model_id.replace(":", "_").replace(".", "_")
        prompt_name_safe = self.prompt_name.replace(" ", "_").lower()
        
        filename = f"extract_results_{model_name_safe}_{prompt_name_safe}.json"
        results_dir = "../documents/extraction_results"
        os.makedirs(results_dir, exist_ok=True)
        results_path = os.path.join(results_dir, filename)
        
        try:
            with open(results_path, 'w', encoding='utf-8') as f:
                json.dump(self.results, f, ensure_ascii=False, indent=4)
            logger.info(f"✅ Resultados salvos em: {results_path}")
        except Exception as e:
            logger.error(f"❌ Falha ao salvar o arquivo JSON: {e}")

In [30]:
multimodal_llms = [("us.anthropic.claude-3-7-sonnet-20250219-v1:0", "bedrock-2023-05-31"),
                   ("us.anthropic.claude-sonnet-4-20250514-v1:0", "bedrock-2023-05-31"),
                   ]


In [31]:
prompts = {
    "prompt_extracao_carteirinha_v1": """
        A imagem fornecida é uma carteirinha de convênio de saúde. Analise a imagem e extraia as seguintes informações em formato JSON:
        - convenio: O nome da operadora do plano de saúde.
        - plano: O tipo ou nome do plano (ex: "Plano Prata", "Enfermaria").
        - nome_pessoa: O nome completo do beneficiário.
        - numero_carteirinha: O número de identificação ou matrícula da carteirinha.

        Se alguma informação não for encontrada, retorne `null` para o campo correspondente.
        O JSON deve ter a seguinte estrutura:
        {
        "convenio": "string",
        "plano": "string",
        "nome_pessoa": "string",
        "numero_carteirinha": "string"
        }
        """
}

In [32]:
# Executar o Experimento de Extração

logger.info("🚀 Configurando e executando o experimento de extração...")
logger.info("=" * 50)

# 1. Definição do Prompt e Nome do Prompt
extraction_prompt = prompts["prompt_extracao_carteirinha_v1"]
prompt_name = "prompt_extracao_carteirinha_v1"


# 2. Verificação dos pré-requisitos
if 'base64_images' in locals() and base64_images:
    try:
        for model_id, model_version in multimodal_llms:
            logger.info(f"Testando o modelo: {model_id, model_version}")
            # 3. Inicialização e Execução do Experimento
            settings = Settings()
            
            experiment = ExtractionExperiment(
                settings=settings,
                model_id=model_id,
                model_version=model_version,  # Use the latest version
                prompt=extraction_prompt,
                prompt_name=prompt_name,
                data_model=CarteirinhaExtraida
            )
            
            experiment.run(base64_images)

    except Exception as e:
        logger.critical(f"❌ Ocorreu um erro crítico durante a configuração do experimento: {e}")
else:
    logger.warning("⚠️ Nenhuma imagem em base64 foi encontrada para processar. Execute as células anteriores para gerar a variável 'base64_images'.")

logger.info("=" * 50)
logger.info("✅ Processo de experimento finalizado.")

{"timestamp": "2025-08-07T11:46:11", "level": "INFO", "name": "__main__", "message": "🚀 Configurando e executando o experimento de extração...", "filename": "1480899590.py", "lineno": 3}
{"timestamp": "2025-08-07T11:46:11", "level": "INFO", "name": "__main__", "message": "==================================================", "filename": "1480899590.py", "lineno": 4}
{"timestamp": "2025-08-07T11:46:11", "level": "INFO", "name": "__main__", "message": "Testando o modelo: ('us.anthropic.claude-3-7-sonnet-20250219-v1:0', 'bedrock-2023-05-31')", "filename": "1480899590.py", "lineno": 15}
{"timestamp": "2025-08-07T11:46:11", "level": "INFO", "name": "__main__", "message": "Criando cliente BEDROCK-RUNTIME para a região: us-east-1...", "filename": "788707453.py", "lineno": 5}
{"timestamp": "2025-08-07T11:46:11", "level": "INFO", "name": "__main__", "message": "Cliente BEDROCK-RUNTIME criado com sucesso.", "filename": "788707453.py", "lineno": 15}
{"timestamp": "2025-08-07T11:46:11", "level": "I

## agentic workflow
### goal here should be the separation of concerns. one orchestrator agent should receive